# Kakunin + OpenAI Swarm Compliance Playground

This notebook demonstrates how to build and execute compliance-guarded AI agents with OpenAI Swarm and Kakunin.

### Step 1: Install Dependencies
First, we'll install `kakunin`, OpenAI `swarm`, `python-dotenv`.

In [ ]:
!pip install kakunin git+https://github.com/openai/swarm.git python-dotenv

### Step 2: Configure Environment Keys
Set your Kakunin and OpenAI API keys below.

In [ ]:
import os
import getpass

os.environ["KAK_API_KEY"] = getpass.getpass("Enter Kakunin API Key (kak_live_...): ")
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key (sk-...): ")

### Step 3: Run the Swarm Agent Demo
Now we'll run a script to register the agent with Kakunin, issue its compliance certificate, bind tool scopes, and run the agent.

In [ ]:
import asyncio
from swarm import Agent
from kakunin import Kakunin
from kakunin.exceptions import ScopeViolationError
from kakunin.integrations.openai_swarm import KakuninSwarm

# Define custom tools
def query_market_prices(symbol: str) -> str:
    """Query current prices for a given ticker symbol."""
    print(f"[Tool Executing] query_market_prices: {symbol}")
    return f"Latest price for {symbol}: $150.00"

def execute_market_trade(symbol: str, amount: int) -> str:
    """Execute a market buy order for a symbol."""
    print(f"[Tool Executing] execute_market_trade: Buying {amount} of {symbol}")
    return f"Successfully executed trade: Buy {amount} shares of {symbol}"

async def run_compliance_demo():
    async with Kakunin(api_key=os.environ["KAK_API_KEY"]) as kakunin_client:
        # 1. Register agent
        agent_record = await kakunin_client.agents.create(
            name="SwarmTrader",
            model="gpt-4o",
            version="1.0.0",
            model_hash="sha256:e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
        )
        print(f"Registered Agent: {agent_record.id}")

        # 2. Issue certificate
        cert = await kakunin_client.agents.certify(agent_record.id)
        print(f"Issued Certificate Serial: {cert.serial_number}")

        # 3. Create compliance Swarm client
        swarm = KakuninSwarm(kakunin=kakunin_client, agent_id=agent_record.id)

        # 4. Initialize Swarm Agent
        agent = Agent(
            name="Compliance Trader",
            instructions="You are a helpful stock trader with pricing and execution tools.",
            functions=[query_market_prices, execute_market_trade],
        )

        tool_scopes_mapping = {
            "query_market_prices": ["market.read"],
            "execute_market_trade": ["trade.execute"]
        }

        print("\n--- Query Price ---")
        try:
            res = swarm.run(
                agent=agent,
                messages=[{"role": "user", "content": "Check price of AAPL"}],
                tool_scopes_mapping=tool_scopes_mapping
            )
            print(f"Response: {res.messages[-1]['content']}")
        except ScopeViolationError as e:
            print(f"Compliance block: {e}")

        print("\n--- Execute Trade ---")
        try:
            res = swarm.run(
                agent=agent,
                messages=[{"role": "user", "content": "Buy 10 shares of AAPL"}],
                tool_scopes_mapping=tool_scopes_mapping
            )
            print(f"Response: {res.messages[-1]['content']}")
        except ScopeViolationError as e:
            print(f"Compliance block: {e}")

await run_compliance_demo()